# Ray Tracing in 15 minutes (JAX Edition)

This is a JAX port of the [PyTorch Ray Tracing notebook](./PyTorch_Ray_Tracing.ipynb), designed to run on TPUs (and GPUs/CPUs).

Ray Tracing is one of the most interesting Computer Graphics algorithms out there, used in [tons of games](https://www.corsair.com/us/en/explorer/gamer/gaming-pcs/what-is-ray-tracing-in-games/) as a default now.

In this notebook, we implement a basic ray tracer with Phong shading using JAX primitives. JAX's functional approach means we use `jnp.where` for conditional assignments instead of in-place masked indexing.

<img src = 'https://drive.google.com/uc?id=1s6qMF0k755WGVp8UuFnzta-ctjBqqssB'>

(created by [Suvaditya Mukherjee](https://suvadityamuk.com))

In [ ]:
import math
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# JAX automatically selects the best available backend (TPU > GPU > CPU)
backend = jax.devices()[0].platform
print(f"Using JAX backend: {backend}")
print(f"Available devices: {jax.devices()}")

In [ ]:
frame_height = 512
frame_width = 512
fov = math.radians(60)

# Try playing around with this by moving it along -z axis!
camera_loc = jnp.array([0.0, 0.0, -1.0])

sphere_loc = jnp.array([0.0, 0.0, -4.0])
sphere_color = jnp.array([0.0, 1.0, 0.0])
sphere_radius = 1.0

light_dir = jnp.array([1.0, 1.0, -1.0])
light_dir = light_dir / jnp.linalg.norm(light_dir)
light_color = jnp.array([1.0, 1.0, 1.0])

specular_exponent = 5.0

ambient_component = 0.1
diffuse_component = 0.3
specular_component = 0.6

# Step 2 - Generate one ray per pixel

Here, we create a `jnp.meshgrid` that yields lists of numbers that we can use to index specific pixel points and/or full rows or columns easily.

<img src = 'https://drive.google.com/uc?id=1ppvmzRvJkKqWc_gCR2l1actiB9eOFNtO'>

In [ ]:
xs, ys = jnp.meshgrid(
    jnp.arange(frame_width, dtype=jnp.float32),
    jnp.arange(frame_height, dtype=jnp.float32),
    indexing="xy",
)

# Step 3 - Map pixel coordinates to camera plane coordinates

We transform pixel coordinates to normalized screen coordinates in the range (-1, 1), where (0, 0) is the center of our camera plane.

- `(X, Y)` -> Range of (frame_width, frame_height)
- `(X + 0.5, Y + 0.5)` -> Maps to center of pixel
- `(tx = (X + 0.5) / frame_width, ty = (Y + 0.5) / frame_height)` -> Normalizes to (0, 1)
- `(2 * tx - 1, 2 * ty - 1)` -> Maps to (-1, 1)

<img src = 'https://drive.google.com/uc?id=1LE6ldLcD2RYCxVIM8oB1imTAjFgS_7qJ'>

In [ ]:
x_screen_normalized = 2 * ((xs + 0.5) / frame_width) - 1
y_screen_normalized = 2 * ((ys + 0.5) / frame_height) - 1

# Step 4 - Set up near plane

We use the field-of-view angle to scale our normalized coordinates to the camera's near/image plane. We also scale the X-axis by the aspect ratio.

<img src = 'https://drive.google.com/uc?id=1TncfRa5ejdX2zbdtUqKw-6I1HO5ztbez'>

In [ ]:
x_camera_plane = x_screen_normalized * math.tan(fov / 2) * (frame_width / frame_height)
y_camera_plane = y_screen_normalized * math.tan(fov / 2)

# Step 5 - Point ray from camera to image plane

A ray is defined by $r = o + (t \cdot d)$. We construct the direction vector for each pixel's ray and normalize it.

We use -1 in the Z-axis because the near plane is 1 unit in the -Z direction from the camera.

<img src = 'https://drive.google.com/uc?id=1eABwrjKBR7ldnfXZJuxm3zmZfCC_v0__'>

In [ ]:
ray_dirs = jnp.stack([
    x_camera_plane,
    y_camera_plane,
    -jnp.ones_like(x_camera_plane)
], axis=-1)

ray_dirs = ray_dirs / jnp.linalg.norm(ray_dirs, axis=-1, keepdims=True)

# Step 6 - Ray-Sphere intersection

To check if a ray intersects a sphere, we substitute the ray equation into the sphere equation $||x - c||^2 = r^2$, yielding a quadratic in $t$:

$(d \cdot d)t^2 + 2(d \cdot m)t + (m \cdot m - R^2) = 0$

where $m = o - c$

<img src = 'https://drive.google.com/uc?id=1EDHWuugxvuUuzbUmsuS2-6M9KOm7t5_-'>

In [ ]:
origin_sphere_center_vec = camera_loc - sphere_loc

b = 2.0 * jnp.sum(ray_dirs * origin_sphere_center_vec, axis=-1)
c = jnp.sum(origin_sphere_center_vec * origin_sphere_center_vec, axis=-1) - sphere_radius ** 2

discriminant = b ** 2 - 4.0 * c

hit_mask = discriminant >= 0.0

# Step 7 - Get valid t from intersection discriminant

We check the discriminant to determine intersection validity:
- Imaginary → no intersection
- Real & positive roots → visible intersection in front of camera
- Real & negative roots → intersection behind camera

In [ ]:
sqrt_discriminant = jnp.sqrt(jnp.clip(discriminant, a_min=0.0))

t0 = (-b - sqrt_discriminant) / 2.0
t1 = (-b + sqrt_discriminant) / 2.0

# Step 8 - Apply t to ray and get intersection point

We pick the correct root, compute intersection points, and derive surface normals.

<img src = 'https://drive.google.com/uc?id=1tvUV8BEq1-nKWo2as7mVKxjM2DMS5hyp'>

In [ ]:
t = jnp.where(t0 > 0, t0, t1)
hit_mask = hit_mask & (t > 0)

intersection_points = camera_loc + ray_dirs * t[..., None]

surface_normal_vecs = intersection_points - sphere_loc
surface_normal_vecs = surface_normal_vecs / jnp.linalg.norm(surface_normal_vecs, axis=-1, keepdims=True)

# Step 9 - Apply Phong Shading

Phong Shading defines 3 color components:

- **Diffuse/Lambertian**: Natural color based on light reflection angle
- **Ambient**: Environment's base light contribution
- **Specular**: Bright highlights from direct light reflection toward the viewer

<img src = 'https://drive.google.com/uc?id=1pL3E-2Cs3sO8ailnrDzbGnvYgblsp5ak'>

In [ ]:
diffuse = jnp.clip(jnp.sum(surface_normal_vecs * (-light_dir), axis=-1), a_min=0.0)

reflected = 2.0 * jnp.sum(surface_normal_vecs * (-light_dir), axis=-1)[..., None] * surface_normal_vecs - (-light_dir)
specular = jnp.clip(jnp.sum(reflected * -(ray_dirs), axis=-1), a_min=0.0) ** specular_exponent

We mix the shading components together in the desired proportions.

In [ ]:
shading = ambient_component + diffuse * diffuse_component

# Step 10 - Render Image and Save

We have our shading ops ready. Now we apply them to get the final image.

**Note:** Unlike PyTorch, JAX arrays are immutable — we cannot do in-place masked assignment like `image[hit_mask] = ...`. Instead, we use `jnp.where` to conditionally blend the sphere color and background.

In [ ]:
# Background color (R, G, B)
bg_r_color = 0.0
bg_g_color = 0.0
bg_b_color = 0.0

bg = jnp.array([bg_r_color, bg_g_color, bg_b_color])
background = jnp.broadcast_to(bg, (frame_height, frame_width, 3))

# Compute sphere pixel colors: object color * shading + light color * specular
sphere_pixel_color = sphere_color * shading[..., None] + light_color * specular[..., None]

# Use jnp.where to select between sphere color and background
# hit_mask is (H, W), expand to (H, W, 1) for broadcasting with (H, W, 3)
image = jnp.where(hit_mask[..., None], sphere_pixel_color, background)

image = jnp.clip(image, a_min=0.0, a_max=1.0)
image_np = np.array((image * 255).astype(jnp.uint8))

final_image = Image.fromarray(image_np)
final_image.save("raytrace_sphere_jax.png")
print("Saved image - raytrace_sphere_jax.png")

# Step 11 - Display Image!

In [ ]:
plt.imshow(image_np)
plt.axis('off')
plt.show()

# Conclusion

In this JAX port, we've replicated the PyTorch ray tracer using JAX primitives (`jax.numpy`). Key differences from the PyTorch version:

1. **No explicit device management** — JAX automatically dispatches to TPU/GPU/CPU
2. **Immutable arrays** — Instead of `image[mask] = value`, we use `jnp.where(mask, value, default)`
3. **`jnp.clip`** instead of `torch.clamp`
4. **`jnp.linalg.norm`** instead of `torch.norm`
5. **NumPy bridge** — We convert to NumPy via `np.array()` for PIL/matplotlib

This notebook runs as-is on TPU, GPU, or CPU — JAX handles the backend selection automatically.

Future resources:

- [Ray Tracing in one weekend](https://raytracing.github.io/books/RayTracingInOneWeekend.html)
- [JAX documentation](https://jax.readthedocs.io/)

Thank you for following along!

If you want to see more, feel free to check out [my website](https://www.suvadityamuk.com), or follow me on [X/Twitter](https://x.com/halcyonrayes), [LinkedIn](https://linkedin.com/in/suvadityamukherjee), and [GitHub](https://github.com/suvadityamuk).

Highly appreciate it if you ⭐ the repository!